# Evaluation of all Models for Species Classifcation

In this notebook I will evaluate all models and explain all approaches I tried in order to increase model accuracy.

In [ ]:
# loading datasets
from species_classification import load_model, create_binary_dataset, train_test_split
import matplotlib.pyplot as plt
import numpy as np

train = np.load("../train.npz")
X_train = train["X"]
y_train = train["y"]

val = np.load("../val.npz")
X_val = val["X"]
y_val = val["y"]

test = np.load("../test.npz")
X_test = test["X"]
y_test = test["y"]

X_train_rgb = X_train[:, :, :, 1:]
X_val_rgb = X_val[:, :, :, 1:]
X_test_rgb = X_test[:, :, :, 1:]

X_train_thermal = X_train[:, :, :, 0:1]
X_val_thermal = X_val[:, :, :, 0:1]
X_test_thermal = X_test[:, :, :, 0:1]

## Data Preprocessing

I cropped the original thermal and RGB images based on the corresponding YOLO annotations. To preserve the image format and to ensure the images are all of the same size, I padded the images to a shape of 128x128. 

## Custom CNN
My first approach was to train a small custom CNN on the thermal images. Unfortunately, it did not lead to good results. 

I tried improving the model performance by applying Data Augmentation, Oversampling, Balanced Class Weights that punish misclassifications of smaller classes more heavily. All of them did either not affect model performance or in some cases even made it worse. 

Data Augmentation in some cases confuses the model more that it helps it learning the actual animal classes. Oversampling did not affect the resulting accuracies much, and the balanced class weights just led to very high loss values because the species are very unevenly distributed. 

Additionally, I tried many different model architectures and hyperparameter combinations that all did not improve the model performance. 

## Custom CNN on Thermal & RGB Images

As training on thermal images only did not result in high accuracy scores, I also downloaded the corresponding RGB images for this dataset. 
Adding the RGB images to the input increased the test accuracy immediately, but unfortunately not much. A possible reason for the only slight improvement is that many labels for the RGB images are misaligned, leading to the animal being not in the center of the cropped image and in some cases even being cut off.

In [ ]:
load_model("../species_class_models/rgb_normal.keras", X_train, y_train, X_val, y_val, X_test, y_test)

## Transfer Learning 

At first I was doing transfer learning with the EfficientNet-TODO on the thermal images. Since EfficientNet was trained on RGB images, it was not really better at classifying the animals than my custom CNN. 

As a result, I tried transfer learning on the RGB images. The problems here are that often the animals are hidden behind trees or because the RGB labels are not correct in some cases, there might not be an animal or only a part of it on the cropped image. 

In [ ]:
load_model("../transfer.keras", X_train_rgb, y_train, X_val_rgb, y_val, X_test_rgb, y_test)

To investigate whether the animals looking too similar to each other causes the low model accuracies or the flights of the different splits being too different, I tried the following two approaches: 

## Binary Classification 
I tried binary classification between deer and pigs. Class 0 are made of Red Deer, Roe Deer, and Fallow Deer. Class 1 consists of Wild Boar and Hybrid Pig. 

I also applied Data Augmentation: 
```python
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])
```

In [ ]:
y_train_bin = create_binary_dataset(y_train)
y_val_bin = create_binary_dataset(y_val)
y_test_bin = create_binary_dataset(y_test)

### Only Thermal Images

In [ ]:
load_model("../species_class_models/binary_thermal.keras", X_train_thermal, y_train_bin, X_val_thermal, y_val_bin, X_test_thermal, y_test_bin)

### Thermal & RGB Images

In [ ]:
load_model("../species_class_models/binary_classification.keras", X_train, y_train_bin, X_val, y_val_bin, X_test, y_test_bin)

## Custom Split
I suspect that the models struggles with classifying images of flights it has never seen before or overall with the very different species distributions in the different datasets. Therefore, I combined the original train, validation, and test datasets and split them customly. I also set the stratify parameter to ensure the same distribution in every dataset.

I also applied Data Augmentation: 
```python
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
```

In [ ]:
X = np.concatenate([X_train_thermal, X_val_thermal, X_test_thermal])
y = np.concatenate([y_train, y_val, y_test])

X_train1, X_temp, y_train1, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val1, X_test1, y_val1, y_test1 = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

load_model("../species_class_models/custom_split.keras", X_train1, y_train1, X_val1, y_val1, X_test1, y_test1)